# Test: Convert a Minutes Paper to Custom XML Schema

General schema format:
```
<committeeDoc>
<documentType> AP <\documentType>
<committeeName> SEN <\committeeName>
<committeeMeetingData> 2024-05-22 </committeeMeetingData>
<committeeYear> 2023/4 </committeeYear>
<meetingNumber> 3 </meetingNumber>
<items>
<agendaItem number="1" paper = "S 23/24 3A">
<metadata> ... </metadata>
<content> ... </content>
<\agendaItem>
     ...
     ...
<\items>
<\committeeDoc>
```

## Imports and file definitions

In [1]:
import pymupdf
from lxml import etree
import re
import pandas as pd
import base64

m_doc = pymupdf.open("SEN_M_20240618.pdf")
ap_doc = pymupdf.open("SEN_AP_20240522.pdf")
sec_doc = pymupdf.open("SEC_AP_20231109.pdf")
temp_filepath = "SenatePapers23-24.csv"

## Understand how blocks are automatically detected

In [2]:
page = m_doc[0]
blocks = page.get_text("dict", sort=True)["blocks"]
for block in blocks:
    if block["type"] == 0:
        # Get the raw text with newlines visible
        spans = [span for line in block["lines"] for span in line["spans"]]
        raw_text = " ".join(s["text"] for s in spans)
        
        print(f"=== BLOCK ===")
        print(f"Raw text repr: {repr(raw_text)}")
        print(f"Num lines: {len(block['lines'])}")
        print(f"Font size: {spans[0]['size'] if spans else 'N/A'}")
        print(f"Bold: {bool(spans[0]['flags'] & 16) if spans else 'N/A'}")
        print()

=== BLOCK ===
Raw text repr: '  Senatus Academicus  Reconvened Meeting '
Num lines: 3
Font size: 10.979999542236328
Bold: True

=== BLOCK ===
Raw text repr: '  Tuesday 18 June 2024 at 9:45-10:45am '
Num lines: 2
Font size: 10.979999542236328
Bold: False

=== BLOCK ===
Raw text repr: 'Microsoft Teams   '
Num lines: 2
Font size: 10.979999542236328
Bold: False

=== BLOCK ===
Raw text repr: 'Confirmed Minute    Attendees:  Peter Adkins, Gill Aitken, Mteeve Amugune, Arianna Andreangeli, Jonathan  Ansell, Kate Ash-Irisarri, Michael Barany, Laura Bickerton, Richard Blythe, Catherine Bovill,  Holly Branigan, Aidan Brown, Rory Callison, Jeremy Carrette, Leigh Chalmers, Neil Chue  Hong, Juan Cruz, Sarah Cunningham-Burley, Sumari Dancer, Luigi Del Debbio, Chris Dent,  Charlotte Desvages, Simone Dimartino, Claire Duncanson, Murray Earle, Tonks Fawcett,  Samantha Fawkner, Manuel Fernandez-Gotz, Chris French, Vashti Galpin, Soledad Garcia  Ferrari, Benjamin Goddard, Richard Gratwick, Colm Harmon, Ga

## Test table extraction

In [3]:
page_with_table = ap_doc[113]
table_blocks = page_with_table.get_text("dict", sort=True)["blocks"]
tables = page_with_table.find_tables()
if tables:
    # 3. Extract the first table
    my_table = tables[0]
    
    # 4. Convert to a readable Pandas DataFrame
    df = my_table.to_pandas()
    print(df)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
                                                Col0 Area of focus  Col2  \
0  Knowledge and\nengagement –\nunderstanding the...          None  None   
1  Research\ngrants/projects –\ncontinuous\nimpro...          None  None   

                                                Col3 Where we are now  Col5  \
0  Key processes and\ndocumentation are\ndifferen...             None  None   
1  Projects not set up\nquickly enough and are\nm...             None  None   

                                                Col6 Where we want to be  \
0  Business processes and\noperating procedures t...                None   
1  We have processes in place\nwhich are simple, ...                None   

   Col8                                               Col9 What we’re doing  \
0  None  • Publish standard operating procedures in\na ...             None   
1  None  • Assess and tackle backlogs to support\nset-u... 

## Create custom blocks

In [ ]:
# Returns true if a block overlaps with any of the tables on a page
def overlaps_tables(block_bbox, table_bboxes):
    bx0, by0, bx1, by1 = block_bbox
    for tx0, ty0, tx1, ty1 in table_bboxes:
        # If the block is entirely outside the table in any direction, no overlap
        if bx1 < tx0 + 1.5 or bx0 > tx1 - 1.5 or by1 < ty0 + 1.5 or by0 > ty1 - 1.5:
            continue
        return True
    return False

# Returns true if a block overlaps with any of the images on a page
def overlaps_imgs(block_bbox, img_bboxes):
    bx0, by0, bx1, by1 = block_bbox
    for ix0, iy0, ix1, iy1 in img_bboxes:
        if bx1 < ix0 + 1.5 or bx0 > ix1 - 1.5 or by1 < iy0 + 1.5 or by0 > iy1 - 1.5:
            continue
        return True
    return False  

# Returns all hyperlinks for a page along with the link's bbox and corresponding text
def get_page_links(page):
    links = []
    for link in page.get_links():
        if link.get("uri"):
            bbox = link["from"]
            display_text = page.get_textbox(bbox).strip()
            links.append({
                "bbox": bbox,
                "uri": link["uri"],
                "display_text": display_text,
            })
    return links

# Returns a dictionary of text spans w/ the matching link -> Used for text blocks
def find_links_in_spans(line_spans, page_links):
    found_links = []
    for span in line_spans:
        sx0, sy0, sx1, sy1 = span["bbox"]
        for link in page_links:
            lx0, ly0, lx1, ly1 = link["bbox"]
            # Check for matching bbox (w/ 1.5 pt tolerance)
            if sx0 >= lx0 - 1.5 and sy0 >= ly0 - 1.5 and sx1 <= lx1 + 1.5 and sy1 <= ly1 + 1.5:
                found_links.append({
                    "text": span["text"], 
                    "uri": link["uri"],    
                })
    return found_links

# Returns a dictionary of display text w/ the matching link -> Used for table cells
def find_links_in_cell(cell_bbox, page_links):
    found_links = []
    cx0, cy0, cx1, cy1 = cell_bbox
    for link in page_links:
        lx0, ly0, lx1, ly1 = link["bbox"]
        # Check to see if link bbox falls completely within cell bbox (w/ 1.5 pt tolerance)
        if lx0 >= cx0 - 1.5 and ly0 >= cy0 - 1.5 and lx1 <= cx1 + 1.5 and ly1 <= cy1 + 1.5:
            found_links.append({
                    "text": link["display_text"], 
                    "uri": link["uri"],    
                })
    return found_links

# Joins all accumulated lines into one block dict
def make_sub_block(current_lines):
    text = " ".join(l["text"] for l in current_lines) 
    text = re.sub(r'\s+', ' ', text).strip() # clean up by getting rid of extra whitespace
    if not text:
        return None  # caller should check for None and skip appending
    return {
        "text": text,
        "font_size": current_lines[0]["font_size"],
        "font": current_lines[0]["font"],
        "bold": current_lines[0]["bold"],
        "bbox": current_lines[0]["bbox"],
        "type": "text",
        "links": [link for l in current_lines for link in l.get("links", [])] # use l.get("links", []) to prevent key error on lines w/o links
    }

# Returns whether a text block is a page number
def is_page_num(block):
    text = block["text"].strip()
    if block["type"] != "text":
        return False
    # Matches page numbers: "1", "12", etc.
    if re.fullmatch(r'\d+', text):
        return True
     # Matches "Page 1 of 9", "Page 12 of 100", etc. 
    if re.fullmatch(r'[Pp]age\s+\d+\s+of\s+\d+', text):
        return True
    return False

# Seperates bullet points and numbered lists in a block
def clean_bullet_points(text):
    text = re.sub(r'\s*•', '\n•', text)
    text = re.sub(r'\s*(?<!\w)o(?!\w)', '\n•', text)
    text = re.sub(r'\s*(\d+\.) ', r'\n\1 ', text) # numbered list
    return text.strip()

# Extract blocks across all pages of a document
def extract_blocks(doc):
    all_blocks = []
    for page in doc:
        page_blocks = []
        page_links = get_page_links(page)
        
        # Extract tables on page and add each cell as a block
        table_bboxes = []
        for table_i, table in enumerate(page.find_tables()):
            table_bboxes.append(table.bbox)
            
            rows = table.extract() # contains a list of lists (rows), where each row contains the text for each cell
            for row_i, row in enumerate(rows):
                for col_i, cell_text in enumerate(row):
                    if not cell_text or not cell_text.strip(): # skip empty cells
                        continue
                    # Get individual cell bbox from table.cells
                    cell_bbox = table.rows[row_i].cells[col_i]
                    
                    cell_text = re.sub(r'\s+', ' ', cell_text).strip() # get rid of extra whitespace
                    cell_text = clean_bullet_points(cell_text)
                    
                    page_blocks.append({
                        "text": cell_text,
                        "type": "table",      
                        "row": row_i,      
                        "col": col_i,  
                        "table_i": table_i,    
                        "page_i": page.number, 
                        "bbox": table.bbox, # bbox for the whole table 
                        "links": find_links_in_cell(cell_bbox, page_links) 
                    })
                        
        # Extract images and add each image as a block
        img_bboxes = []
        for img in page.get_images():
            xref = img[0]
            
            rects = page.get_image_rects(xref) # gets all bboxes where that image occurs
            if not rects:
                continue
                
            # Take first instance of image as bbox (there should only be 1)
            bbox = rects[0] 
            img_bboxes.append(bbox)
            
            # Extract the image's raw data 
            raw_img = doc.extract_image(xref)
            if raw_img:
                img_bytes = raw_img["image"]  # Raw binary data
                img_ext = raw_img["ext"]      # e.g., 'png', 'jpeg'
                
                # Convert the binary bytes into a Base64 string -> image will live entirely in the XML
                base64_encoded = base64.b64encode(img_bytes).decode("utf-8")
                
                page_blocks.append({
                    "type": "img",
                    "bbox": list(bbox), # need to convert rect object to list
                    "image_data": base64_encoded,
                    "image_ext": img_ext
                })
        
        # Extract clean text blocks and add them
        blocks = page.get_text("dict", sort = True)["blocks"]
        for block in blocks:
            # Only look at text blocks (type 0)
            if block["type"] != 0:
                continue

            # Skip block if it overlaps with tables or images (content already captured)
            if overlaps_tables(block["bbox"], table_bboxes) or overlaps_imgs(block["bbox"], img_bboxes):
                continue
            
            # Flatten all spans in the block to ensure the block actually contains text
            spans = [span for line in block["lines"] for span in line["spans"]]
            if not spans: 
                continue
            
            # Build a metadata dictionary per line
            lines_meta = []
            for line in block["lines"]:
                line_spans = line["spans"]
                if not line_spans:
                    continue
                
                # Get rid of unnecessary heading frequently at the top of SEN papers (needs to be done in the spans level)
                line_spans = [span for span in line_spans if span['text'].strip() != 'H/02/02/02']
                if not line_spans:
                    continue
                    
                # Combine text across all spans for the line
                line_text = " ".join(span["text"] for span in line_spans)
                
                # Identify the 'dominant' span (the longest text piece) to represent the line's style
                line_dominant = max(line_spans, key=lambda s: len(s["text"])) 
                
                lines_meta.append({
                    "text": line_text,
                    "bbox": line_dominant["bbox"],
                    "font": line_dominant["font"],
                    "font_size": line_dominant["size"],
                    "bold": bool(line_dominant["flags"] & 16), # Bitwise check for bold flag
                    "type": "text",
                    "links": find_links_in_spans(line_spans, page_links) 
                })

            # Group lines into sub-blocks based on formatting continuity
            sub_blocks = []
            current_lines = [] 
            
            for line in lines_meta:
                if not current_lines:
                    current_lines.append(line)
                    continue

                prev_line = current_lines[-1]
                should_split = False

                # Split conditions: change in size, font, bold state, or large whitespace gaps
                if abs(line["font_size"] - prev_line["font_size"]) > 1:
                    should_split = True
                elif line["font"] != prev_line["font"]:
                    should_split = True
                elif line["bold"] != prev_line["bold"]:
                    should_split = True
                elif re.search(r'\n{2,}', line["text"]):
                    should_split = True
                elif re.search(r'\s{3,}', line["text"]):
                    should_split = True

                if should_split:
                    new_block = make_sub_block(current_lines)
                    if new_block is not None:
                        sub_blocks.append(new_block)
                    current_lines = [line] # Reset and start a new sub-block with the current line
                else:
                    current_lines.append(line) # Append to the ongoing sub-block

            # Add the final accumulated group of lines
            if current_lines:
                text = " ".join(l["text"] for l in current_lines)
                text = re.sub(r'\s+', ' ', text).strip()
                if text:
                    new_block = make_sub_block(current_lines)
                    if new_block is not None:
                        sub_blocks.append(new_block)

                # Save valid sub-blocks to the global page blocks list
                for b in sub_blocks:
                    if b["text"]:
                        page_blocks.append(b)
                        
        # Sort blocks in page to appear by position 
        page_blocks.sort(key=lambda b: b["bbox"][1]) # bbox[1] = y0
        
        # Put newlines where there are bulletpoints ("•" or "o")
        for block in page_blocks:
            if block["type"] != "img":
                block["text"] = clean_bullet_points(block["text"])
        
        all_blocks.extend(page_blocks)
                
                
    # Merge blocks that are part of the same paragraph together
    extracted_blocks = []
    buffer = None
    seen_images = set()

    for block in all_blocks:
        # If we encounter an image, clear out the buffer
        if block["type"] == "img":
            # Skip duplicate images throughout doc
            if block["image_data"] in seen_images:
                continue
            seen_images.add(block["image_data"])
            
            if buffer:
                extracted_blocks.append(buffer)
                buffer = None
            extracted_blocks.append(block)
            continue
            
        if is_page_num(block): # skip page nums for XML conversion
            continue
        
        curr_text = block["text"].strip()
            
        # If there's no active buffer, make this block the buffer
        if buffer is None:
            buffer = block
            continue

        # Check for paragraph continuation rules
        same_sentence = False
        if curr_text:
            lowercase_start = curr_text[0].islower()
            same_font_size = True
            if block['type'] == 'text' and buffer['type'] == 'text':
                same_font_size = not abs(block["font_size"] - buffer["font_size"]) > 1
            same_sentence = lowercase_start and same_font_size
            
        list_item = re.fullmatch(r'\d+\.', buffer["text"])
        bullet_point = re.fullmatch(r'•', buffer["text"])
        same_paragraph = same_sentence or list_item or bullet_point
        
        if same_paragraph:
            buffer["text"] = buffer["text"] + " " + curr_text
        else:
            extracted_blocks.append(buffer)
            buffer = block

    if buffer:
        extracted_blocks.append(buffer)
    
    return extracted_blocks

## Extracting metadata
The following code extracts metadata from a CSV file. If the file is not found within the CSV by its filename, then it is instead extracted automatically from the filename. However, as meeting number cannot be inferred from file name, it is left as `Unknown`.

Metadata:
* Document type
* Committee name
* Meeting date (or meeting start date and end date)
* Committee year
* Meeting number

Notes:
* Another solution is to have start date be the same as end date for single day meetings
* Meeting number cannot automatically be extracted for minutes. For agenda and papers, it can be deduced from the paper codes. But minutes do not generally include paper codes (except in reference to past/future meetings), which would be risky to extract incorrectly
* We are currently assuming that August 1st begins a new academic year

In [5]:
# Extract metadata automatically from CSV
def fetch_metadata_from_csv(target_file, csv_path):
    df = pd.read_csv(csv_path)
    match = df[df['fileName'] == target_file] # no ext (such as .pdf)
    if match.empty:
        return None
    else:
        return match.iloc[0].to_dict() 

# Extract metadata automatically from filename (does not extract meeting number)
def extract_metadata_from_filename(target_file):
    parts = target_file.split('_')
    e_meeting = len(parts) > 3
    
    # Extract doc type
    extracted_doc_type = parts[1]

    # Extract committee name
    extracted_committee_name = None
    if parts[0] == "SEN": #in CSV, Senate and E-Senate do not keep abbreviation
        if e_meeting:
            extracted_committee_name = "E-Senate"
        else:
            extracted_committee_name = "Senate"
    else:
        extracted_committee_name = parts[0]
        
    # Extract meeting date(s)
    extracted_start_date = None
    extracted_end_date = None

    if e_meeting: # seperate start and end dates
        extracted_start_date = parts[2]
        extracted_end_date = parts[3]
    else:
        extracted_start_date = extracted_end_date = parts[2]
    
    # Extract academic year 
    year = int(extracted_start_date)[:4] 
    new_academic_year = f"{year}0801" # New academic year starts August 1 of the year
    extracted_academic_year = f"{year}/{year+1}" if extracted_start_date >= new_academic_year else f"{year-1}/{year}"
    
    def fix_date_formatting(d): # DD/MM/YYYY
        if d:
            return f"{d[6:]}/{d[4:6]}/{d[:4]}"
        return None
    extracted_start_date, extracted_end_date = map(fix_date_formatting, (extracted_start_date, extracted_end_date))
    
    return {
        "fileName": target_file,
        "documentType": extracted_doc_type,
        "committeeName": extracted_committee_name,
        "startDate": extracted_start_date,
        "endDate": extracted_end_date,
        "academicYear": extracted_academic_year,
        "meetingNumber": "Unknown"
    }

# Extract metadata
def extract_metadata(file, csv_filepath):
    filename = file.name.removesuffix('.pdf')
    
    # Validate filename
    valid_meeting_pattern = r"^(SEN|SEC|APRC|SQAC)_(AP|M)_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])$" #supports years 2000-2099
    valid_e_meeting_pattern = r"^(SEN|SEC|APRC|SQAC)_(AP|M)_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])_20\d{2}(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])_e$"
    if not(re.match(valid_meeting_pattern, filename) or re.match(valid_e_meeting_pattern, filename)):
        raise Exception("Invalid filename format")
    
    # Try extraction from CSV first
    metadata = fetch_metadata_from_csv(filename, csv_filepath)
    
    # Fallback to data extraction from filename
    if metadata is None:
        metadata = extract_metadata_from_filename(filename)
        
    return metadata
    
print(extract_metadata(m_doc, temp_filepath))


{'fileName': 'SEN_M_20240618', 'documentType': 'M', 'committeeName': 'Senate', 'startDate': '18/6/2024', 'endDate': '18/6/2024', 'academicYear': '2023/2024', 'meetingNumber': 4}


## Create XML document
Note:
* Including just 1 tag for date; if over multiple days, included in the following format: "2024-06-10 to 2024-06-18"

In [9]:
# Creates a <tag> element under parent_el
    # Embeds links as <a> inline
    # Creates a <br> tag before each bullet point so it displays on a new line
def build_text_el(parent_el, tag, block):
    el = etree.SubElement(parent_el, tag)
    links = block.get("links", [])
    
    # Split on newlines —> marks where bullet point begins
    parts = block["text"].split("\n")
    
    for i, part in enumerate(parts):
        part = part.strip()
        if not part:
            continue
        
        if i > 0: # no need for an empty newline before the first bit of text
            br = etree.SubElement(el, "br")
        
        # For each part, find which links fall within it
        part_links = [l for l in links if l["text"] in part]
        
        # No links, set text directly
        if not part_links:
            if i == 0: # no <br> beforehand
                el.text = (el.text or "") + part # or "" handles when el.text = None
            else:
                br.tail = part
        # Build inline <a> tags
        else: 
            remaining_text = part
            
            # Whether the text before the first link goes in el.text or br.tail
            is_br_target = i > 0
            
            for link in part_links:
                link_text = link["text"]
                uri = link["uri"]
                
                # Find where the linked text sits within the remaining text
                link_i = remaining_text.find(link_text)
                if link_i == -1:
                    continue
                
                before = remaining_text[:link_i]
                after = remaining_text[link_i + len(link_text):]
                
                # Text before the link
                if len(el) == 0:
                    el.text = (el.text or "") + before 
                else:
                    el[-1].tail = (el[-1].tail or "") + before # el[-1] is the last <a> tag
                
                # Create the inline <a>
                a = etree.SubElement(el, "a")
                a.set("href", uri)
                a.text = link_text
                a.tail = ""
                
                remaining_text = after
            
            # Remaining text after the last link
            el[-1].tail = (el[-1].tail or "") + remaining_text
    
    return el

# Converts pdf file to xml file 
def convert_pdf_to_xml(file, csv_filepath):
    root = etree.Element("committeeDoc")
    tree = etree.ElementTree(root)
    metadata = extract_metadata(file, csv_filepath)

    # Connect to the XSLT styling sheet
    xslt = etree.ProcessingInstruction(
        "xml-stylesheet", 'type="text/xsl" href="style.xslt"'
    )
    root.addprevious(xslt)

    # Add metadata (docType, committeeName, dates, academicYear)
    metadata_el = etree.SubElement(root, "metadata")
    doc_type = etree.SubElement(metadata_el, "documentType")
    doc_type.text = metadata["documentType"]
    committee_name = etree.SubElement(metadata_el, "committeeName")
    committee_name.text = metadata["committeeName"]
    start_date = etree.SubElement(metadata_el, "startDate")
    start_date.text = metadata["startDate"]
    end_date = etree.SubElement(metadata_el, "endDate")
    end_date.text = metadata["endDate"]
    academic_year = etree.SubElement(metadata_el, "academicYear")
    academic_year.text = metadata["academicYear"]
    meeting_num = etree.SubElement(metadata_el, "meetingNumber")
    meeting_num.text = str(metadata["meetingNumber"])
    
    # First item will always be the agenda, then the papers
    agenda_el = etree.SubElement(root, "agenda")

    # Loop through extracted blocks
    # Current identification --> bold = heading, anything else = bodyText
    current_item = agenda_el
    current_table = None
    current_table_i = -1
    current_row = None
    current_row_i = -1
    current_page_i = -1
    current_paper_code = None
    for block in extract_blocks(file):
        type = block["type"]
        
        if type == "img":
            img_el = etree.SubElement(current_item, "image", ext = block["image_ext"])
            img_el.text = block["image_data"]
        
        elif type == "table":
            # Create a new table when either the page or the table index changes
            is_new_table = (
                block["page_i"] != current_page_i or
                block["table_i"] != current_table_i
            )

            if current_table is None or is_new_table:
                current_table = etree.SubElement(current_item, "table")
                current_row = None
                current_row_i = -1
                current_table_i = block["table_i"]
                current_page_i = block["page_i"]

            # Create a new row when the row index changes
            if block["row"] != current_row_i:
                current_row = etree.SubElement(current_table, "row")
                current_row_i = block["row"]

            build_text_el(current_row, "cell", block)
        
        elif type == "text":
            text = block["text"]
            font_size = block["font_size"]
            font = block["font"]
            bold = block["bold"]
            
            # Reset table
            current_table = None
            current_table_i = -1
            current_row = None
            current_row_i = -1
            current_page_i = -1
            
            if font_size > 12: # paper code identified by large font size
                # Only start a new paper if the paper code is different from the current one
                if current_paper_code == block["text"]:
                    continue
                else:
                    paper_el = etree.SubElement(root, "paper", paperCode = block["text"])
                    current_item = paper_el
                    current_paper_code = block["text"]
            elif bold:
                build_text_el(current_item, "boldText", block)
            else:
                build_text_el(current_item, "bodyText", block)   

    # Save to same directory
    xml_filename = f"{file.name.removesuffix('.pdf')}.xml"
    with open(xml_filename, "wb") as f:
        tree.write(f, xml_declaration=True, encoding="utf-8", pretty_print=True)
    return  

# View tree so far
convert_pdf_to_xml(ap_doc, temp_filepath)

# TESTING: identifying paper codes
* Generally, the paper code is in the upper right corner of the document, much larger than other text, and seems to be bold
* Sometimes the paper code is only on the first page of the paper
* Sometimes the paper code will not be as large after the first page
* Sometimes the paper code will be in the middle of the page if the format switches to landscape

Generally the body text is around font size 11-12 while the paper code is around

Issue: sometimes there is a little bit of text "H/02/02/02" which is on the same line as the paper code but tiny and so the way it is extracted now it looks like body text --> check where I am eliminating this b/c it should be in spans

In [7]:
test_pages = sec_doc[34:35]

for block in extract_blocks(test_pages):
    if block['type'] == 'img':
        print("IMAGE GOES HERE")
    elif block['type'] == 'table':
        print(block['text'])
        print()
        print(f"Row: {block['row']}, Column: {block['col']}")
    else:
        print(block['text'])
        print()
        print(f"Font size: {block['font_size']}")
    print("---")
    
# pages w/ paper codes: 4, 25, 29, 63, 93, 105, 

SEC 23/24 2G

Font size: 14.039999961853027
---
• Share insights to inform the development of an asynchronous version of the Edinburgh Award. This feedback has been provided to the proposer, was well received and will be actioned by them.

Font size: 12.0
---
Resource implications

Font size: 12.0
---
9. There will be workload implications for the Mastercard Foundation Scholar Program staff responsible for managing and verifying the activity. Some development work by Student Systems will be required to add the new activity to the HEAR.

Font size: 12.0
---
Risk management

Font size: 12.0
---
10. N/A

Font size: 12.0
---
Responding to the Climate Emergency & Sustainable Development Goals

Font size: 12.0
---
11. N/A

Font size: 12.0
---
Equality & diversity

Font size: 12.0
---
12. The proposed activity is open to the 850 taught postgraduate students on the program, the majority being online distance learning students going forward. Although there is an Edinburgh Award that can be take

# TESTING: sometimes it associates body text in the same block as the paper code
* Correctly detects that it is a different line

In [8]:
testing = sec_doc[27:29]

# for block in extract_blocks(test_pages):
#     if block['type'] == 'img':
#         print("IMAGE GOES HERE")
#     elif block['type'] == 'table':
#         print(block['text'])
#         print()
#         print(f"Row: {block['row']}, Column: {block['col']}")
#     else:
#         print(block['text'])
#         print()
#         #print(f"Font size: {block['font_size']}")
#     print("---")
    
for page in testing:
    blocks = page.get_text("dict", sort = True)["blocks"]
    for block in blocks:
        # Only look at text blocks (type 0)
        if block["type"] != 0:
            continue

        # Flatten all spans in the block to ensure the block actually contains text
        spans = [span for line in block["lines"] for span in line["spans"]]
        if not spans: 
            continue
        
        # Build a metadata dictionary per line
        lines_meta = []
        for line in block["lines"]:
            line_spans = line["spans"]
            if not line_spans:
                continue
            
            # Get rid of unnecessary heading frequently at the top of SEN papers (needs to be done in the spans level)
            line_spans = [span for span in line_spans if span['text'].strip() != 'H/02/02/02']
            if not line_spans:
                continue
                
            # Combine text across all spans for the line
            line_text = " ".join(span["text"] for span in line_spans)
            
            # Identify the 'dominant' span (the longest text piece) to represent the line's style
            line_dominant = max(line_spans, key=lambda s: len(s["text"])) 
            
            print("SEPARATE LINE")
            print(line_text)
            print(f"Font size: {line_dominant['size']}")
            print()
            
            # lines_meta.append({
            #     "text": line_text,
            #     "bbox": line_dominant["bbox"],
            #     "font": line_dominant["font"],
            #     "font_size": line_dominant["size"],
            #     "bold": bool(line_dominant["flags"] & 16), # Bitwise check for bold flag
            #     "type": "text",
            #     "links": find_links_in_spans(line_spans, page_links) 
            # })

SEPARATE LINE
SEC 23/24 2C 
Font size: 14.039999961853027

SEPARATE LINE
responsibilities; mandatory induction and training; non-mandatory training and 
Font size: 12.0

SEPARATE LINE
development; and resolving problems.  
Font size: 12.0

SEPARATE LINE
  
Font size: 12.0

SEPARATE LINE
Other learning and teaching policies   
Font size: 12.0

SEPARATE LINE
 
Font size: 12.0

SEPARATE LINE
Open Educational Resources Policy  (pdf)  
Font size: 12.0

SEPARATE LINE
 
Font size: 12.0

SEPARATE LINE
This policy outlines the University’s position on Open Educational Resources (OERs) and 
Font size: 12.0

SEPARATE LINE
provides guidelines for practice in learning and teaching.  
Font size: 12.0

SEPARATE LINE
 
Font size: 12.0

SEPARATE LINE
Lecture Recording Policy  (pdf)  
Font size: 12.0

SEPARATE LINE
 
Font size: 12.0

SEPARATE LINE
This policy has been developed to ensure that: Provision of recorded lectures is 
Font size: 12.0

SEPARATE LINE
comprehensive, consistent and efficient and e